# From PyTorch to MNIST

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/05-pytorch/02-From-PyTorch-to-MNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Recommended environment: Google Colab with a T4 GPU (`Runtime → Change runtime type → T4 GPU`).

In [ ]:
# Download the reference solutions for this notebook's exercises (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/05-pytorch/'
for fn in ['q_softmax.py', 'q_mlp.py', 'q_cnn.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
from IPython.display import display
device = "cuda" if torch.cuda.is_available() else "cpu"
device

## Loading MNIST

`torchvision.datasets` includes many common datasets. Here we load all of MNIST directly as tensors (it is only 60000x28x28, so it fits on the GPU easily).

In [ ]:
from torchvision.datasets import MNIST
train_ds = MNIST("data", train=True, download=True)
test_ds = MNIST("data", train=False, download=True)
train_X = train_ds.data.float().reshape(-1, 784) / 255
train_y = train_ds.targets
test_X = test_ds.data.float().reshape(-1, 784) / 255
test_y = test_ds.targets
train_X.shape, train_y.shape, test_X.shape

In [ ]:
def showX(X):
    int_X = (X * 255).clamp(0, 255).to(torch.uint8).cpu().numpy()
    int_X_reshape = int_X.reshape(-1, 28, 28).swapaxes(0, 1).reshape(28, -1)
    display(Image.fromarray(int_X_reshape))
showX(train_X[:20])
print(train_y[:20])

## Logistic Regression from scratch

Model: $\hat y = \mathrm{softmax}(XW + b)$

Loss: cross entropy

In [ ]:
W = torch.zeros(784, 10, requires_grad=True)
b = torch.zeros(10, requires_grad=True)
def model(X):
    return X @ W + b     # outputs logits, before softmax

## Q
Write your own softmax and cross entropy, and compare the result with `F.cross_entropy`.
* $\mathrm{softmax}(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$
* cross entropy $=$ the mean of $-\log \mathrm{softmax}(z)_{y}$
* Hint: for numerical stability, subtract the maximum first, or just use `torch.logsumexp`

In [ ]:
# %load q_softmax.py
%run -i q_softmax.py

### A hand-written training loop (mini-batch SGD)

In [ ]:
lr = 0.5
batch_size = 256
for epoch in range(5):
    perm = torch.randperm(len(train_X))
    for i in range(0, len(train_X), batch_size):
        idx = perm[i:i+batch_size]
        loss = F.cross_entropy(model(train_X[idx]), train_y[idx])
        loss.backward()
        with torch.no_grad():
            W -= lr * W.grad
            b -= lr * b.grad
            W.grad.zero_(); b.grad.zero_()
    with torch.no_grad():
        acc = (model(test_X).argmax(dim=1) == test_y).float().mean()
    print(f"epoch {epoch} loss={loss.item():.4f} test acc={acc:.4f}")

In [ ]:
# Let's see what W looks like
showX((W.detach().T - W.min()) / (W.max() - W.min()))

## Rewriting with `nn.Module` and `torch.optim`

In real code you use ready-made layers such as `nn.Linear`, a `DataLoader` to fetch batches, and an optimizer to update the parameters.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
train_loader = DataLoader(TensorDataset(train_X, train_y), batch_size=256, shuffle=True)
test_loader = DataLoader(TensorDataset(test_X, test_y), batch_size=1024)

In [ ]:
def train(model, epochs=5, lr=1e-3):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            loss = F.cross_entropy(model(X), y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(f"epoch {epoch} loss={loss.item():.4f} test acc={evaluate(model):.4f}")

@torch.no_grad()
def evaluate(model):
    model.eval()
    correct = 0
    for X, y in test_loader:
        X, y = X.to(device), y.to(device)
        correct += (model(X).argmax(dim=1) == y).sum().item()
    return correct / len(test_loader.dataset)

In [ ]:
model = nn.Linear(784, 10)
train(model)

In [ ]:
# Predict the first 20 samples of test_X
with torch.no_grad():
    print(model(test_X[:20].to(device)).argmax(dim=1))
showX(test_X[:20])

## Q
Build a Multi-Layer Perceptron:
* `Linear(784, 256)` → `ReLU` → `Linear(256, 10)`
* Use `nn.Sequential`
* Try adding `nn.Dropout` and more layers

In [ ]:
# %load q_mlp.py
%run -i q_mlp.py

### Custom `nn.Module`

More complex models are usually written as a class, with the computation described in `forward`.

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden=512):
        super().__init__()
        self.fc1 = nn.Linear(784, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, 10)
    def forward(self, x):
        x = F.gelu(self.fc1(x))
        x = F.gelu(self.fc2(x)) + x   # residual connection
        return self.fc3(x)
model = MLP()
print(model)
print("parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
train(model)

## Q: Convolutional Neural Network
Build the following CNN and train it:
* Reshape to `(N, 1, 28, 28)` (PyTorch is channels-first)
* `Conv2d(1, 32, kernel_size=3, padding=1)` → `ReLU` → `MaxPool2d(2)`
* `Conv2d(32, 64, kernel_size=3, padding=1)` → `ReLU` → `MaxPool2d(2)`
* `Flatten` → `Linear(64*7*7, 1024)` → `ReLU` → `Dropout(0.5)` → `Linear(1024, 10)`

Hint: `nn.Unflatten(1, (1, 28, 28))` can be the first layer of the Sequential. Accuracy should exceed 99%.

In [ ]:
# %load q_cnn.py
%run -i q_cnn.py

## Saving and loading

Save the `state_dict` (parameters only) rather than the whole model object.

In [ ]:
torch.save(model.state_dict(), "mnist_cnn.pt")
model2 = cnn_model() if "cnn_model" in globals() else MLP()
model2.load_state_dict(torch.load("mnist_cnn.pt"))
model2.to(device)
evaluate(model2)